# W3D5 — Tuning an MLP on MNIST, and the Week's Reckoning — Lab

**Week 3 · Day 5 · Deep Learning & Neural Networks** · Lab

The data changes today, for the first time since Tuesday. **MNIST**: 28×28 greyscale images of
handwritten digits, loaded with `torchvision` — pictures instead of rows, ten classes instead of two.
Everything else is the week's machinery, unchanged: an `nn.Module`, a `DataLoader`, the five-line loop.

Two halves, and they have different jobs.

The first is **diagnosis**. Four loss curves, four different faults, and the vocabulary for naming
them on sight: a learning rate too high, a learning rate too low, overfitting, and a model that has
learned nothing at all and says so with the number 2.3026 — `ln 10`, the flat line of a ten-class
problem. Then you reproduce the overfitting curve on purpose, and try the standard fixes on it —
dropout, early stopping, a different optimiser — and record which of them actually worked on **this**
data rather than which one is supposed to.

The second is **the reckoning**. On Monday the best model on a table was a boosted tree. Today the same
kind of tree gets the same pixels your network gets, both are scored once on 10,000 test images they
have never seen, and you report what happened, with both spreads.

You leave with `experiments.parquet` and `best_model.pt`. **The capstone brief is released today**,
and `experiments.parquet` is the shape of the experiment log it asks for.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٣ اليوم ٥ — ضبط الشبكة على MNIST، وحساب الأسبوع

**الأسبوع ٣ · اليوم ٥ · التعلّم العميق والشبكات العصبية** · معمل عملي

تتغيّر البيانات اليوم لأول مرة منذ الثلاثاء. **MNIST**: صور رمادية بحجم ٢٨×٢٨ لأرقام مكتوبة بخط اليد،
تُحمَّل بـ `torchvision` — صور بدل صفوف، وعشر فئات بدل اثنتين. وكل ما عدا ذلك هو آلة الأسبوع بلا تغيير:
`nn.Module` و`DataLoader` وحلقة الأسطر الخمسة.

نصفان، ولكلٍّ منهما عمل مختلف.

الأول **التشخيص**: أربعة منحنيات خسارة وأربعة أعطال مختلفة، والمفردات التي تسمّيها بها من النظرة
الأولى: معدّل تعلّم مرتفع، ومعدّل منخفض، وفرط مطابقة، ونموذج لم يتعلّم شيئًا ويقول ذلك بالرقم ٢٫٣٠٢٦ —
أي `ln 10`، الخط المسطّح لمسألة من عشر فئات. ثم تُعيد إنتاج منحنى فرط المطابقة عن قصد، وتُجرّب عليه
العلاجات المعيارية — Dropout والإيقاف المبكر ومُحسِّنًا آخر — وتسجّل أيّها نفع على **هذه** البيانات لا
أيّها يُفترض أن ينفع.

والثاني **الحساب**. فقد كانت أفضل نموذج على جدول الاثنين شجرةٌ معزَّزة. واليوم يأخذ النوع نفسه من
الأشجار البكسلات نفسها التي تأخذها شبكتك، ويُقاس الاثنان مرة واحدة على ١٠٬٠٠٠ صورة اختبار لم يرياها،
وتُبلّغ بما حدث بمدى تفاوت كلٍّ منهما.

ستخرج بملفَّي `experiments.parquet` و`best_model.pt`. **وتُنشر كرّاسة مشروع التخرّج اليوم**،
و`experiments.parquet` هو شكل سجلّ التجارب الذي تطلبه.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Load an image dataset with `torchvision`, say what `ToTensor` and `Normalize` do to a pixel, and read
  an image batch's shape as `(batch, channels, height, width)`.
- Feed images to an MLP with `nn.Flatten`, and train a ten-class classifier with
  `nn.CrossEntropyLoss` on raw logits.
- Diagnose a training run from its loss curve, and name the fix: divide the learning rate by ten,
  raise it, stop earlier, or go and look at the model and the data.
- Recognise `ln 10 = 2.3026` on a balanced ten-class problem — and `ln 2` on a binary one — and know it
  means the model has learned nothing.
- Produce overfitting on purpose, and measure whether dropout fixed it on your data.
- Compare four optimisers under identical seeds, and prove they were identical by their initial loss.
- Implement early stopping that restores the **best** weights, and report the two numbers either side
  of it.
- Keep an experiment log where every row differs from another by exactly one hyperparameter.
- Save a model with its config, reload it, and reproduce its recorded score exactly.
- Score on a test set exactly once, and put a neural network next to a boosted tree with both spreads
  without flattering either.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- تحميل مجموعة صور بـ `torchvision`، وبيان ما يفعله `ToTensor` و`Normalize` بالبكسل، وقراءة شكل دفعة
  الصور على أنه `(batch, channels, height, width)`.
- تمرير الصور إلى شبكة متعدّدة الطبقات بـ `nn.Flatten`، وتدريب مصنّف من عشر فئات بـ
  `nn.CrossEntropyLoss` على الـ logits الخام.
- تشخيص تدريب من منحنى خسارته وتسمية العلاج: اقسم معدّل التعلّم على عشرة، أو ارفعه، أو أوقِف مبكرًا،
  أو اذهب وانظر في النموذج والبيانات.
- التعرّف على `ln 10 = 2.3026` في مسألة متوازنة من عشر فئات — و`ln 2` في مسألة ثنائية — ومعرفة أنه
  يعني أن النموذج لم يتعلّم شيئًا.
- إنتاج فرط المطابقة عن قصد، وقياس هل عالجه Dropout على بياناتك.
- مقارنة أربعة مُحسِّنات ببذور متطابقة، وإثبات تطابقها بخسارتها الابتدائية.
- تنفيذ إيقاف مبكر يستعيد **أفضل** الأوزان، وعرض الرقمين على جانبيه.
- حفظ سجلّ تجارب يختلف فيه كل صف عن غيره بمعامل واحد بالضبط.
- حفظ نموذج مع إعداده وإعادة تحميله وإعادة إنتاج نتيجته المسجّلة بالضبط.
- القياس على مجموعة الاختبار مرة واحدة فقط، ووضع شبكة عصبية بجانب شجرة معزَّزة بمدى تفاوت كلٍّ منهما
  بلا مجاملة لأيٍّ منهما.

</div>


## About the data

**Dataset:** `mnist` — 70,000 greyscale images of handwritten digits, 28×28 pixels, ten balanced
classes, CC BY-SA 3.0. `torchvision` downloads it (about 11 MB) the first time you run the setup and
caches it in this lab's `artefacts/` folder.

An image is not a row, and three things follow from that:

- **It arrives as a picture.** On disk each digit is a 28×28 grid of integers from 0 to 255. A
  **transform** turns it into what a network eats: `ToTensor` makes a `(1, 28, 28)` float tensor in
  `[0, 1]` — one channel, because it is greyscale — and `Normalize` subtracts MNIST's mean pixel and
  divides by its spread, so the inputs are centred on zero the way Tuesday's He initialisation assumes.
- **An MLP throws the picture away.** `nn.Flatten` turns `(1, 28, 28)` into 784 numbers in a row, and
  from then on the network has no idea which pixels were neighbours. It still works — surprisingly
  well — and next week's convolutional networks are what you get when you stop throwing that
  structure away.
- **The label is a class index.** `y` holds integers 0 to 9, not one-hot vectors, and
  `nn.CrossEntropyLoss` takes them as they are.

Three splits, and each one has one job:

- **Training: 1,000 images**, drawn from the 60,000-image training set. Deliberately few. A network with
  270,000 parameters memorises 1,000 images within a handful of epochs, and overfitting is the thing
  this lab has to have in front of it. It also keeps every run to a few seconds on a laptop CPU.
- **Validation: 5,000 other training images.** Every decision today — dropout, optimiser, learning
  rate, when to stop — is made on these.
- **Test: the official 10,000.** Touched once, at the very end, in task 2.6. A number you tuned
  against is not a test score.

You also load **`tree_baseline.json`** from Monday, for the week's comparison. If you did not finish
that lab, `load_artefact` falls back to the reference copy and prints a note.

**Watch out:** MNIST is unrealistically easy — a convolutional network passes 99% on it, and a
near-perfect score here says nothing about real vision work. With only 1,000 training images an MLP
lands around 90%, and that is the honest number for this setup. Do not compare it with a leaderboard
that trained on all 60,000.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `mnist` — ٧٠٬٠٠٠ صورة رمادية لأرقام مكتوبة بخط اليد، بحجم ٢٨×٢٨ بكسلًا، وعشر فئات
متوازنة، رخصة CC BY-SA 3.0. يُنزّلها `torchvision` (نحو ١١ ميغابايت) أول مرة تشغّل فيها الإعداد
ويخزّنها في مجلّد `artefacts/` لهذا المعمل.

والصورة ليست صفًّا، ويترتّب على ذلك ثلاثة أمور:

- **تصل صورةً.** فكل رقم على القرص شبكة ٢٨×٢٨ من أعداد صحيحة بين ٠ و٢٥٥. و**التحويل** (transform) يجعله
  ما تأكله الشبكة: فـ `ToTensor` يصنع تنسورًا عشريًا بشكل `(1, 28, 28)` في المدى `[0, 1]` — بقناة واحدة
  لأنه رمادي — و`Normalize` يطرح متوسط بكسل MNIST ويقسم على تشتّته، فتتمركز المُدخلات حول الصفر كما
  تفترض تهيئة He يوم الثلاثاء.
- **والشبكة متعدّدة الطبقات ترمي الصورة.** فـ `nn.Flatten` يحوّل `(1, 28, 28)` إلى ٧٨٤ رقمًا في صف، ومن
  بعدها لا تعرف الشبكة أي البكسلات كانت متجاورة. وهي تعمل مع ذلك — بجودة مفاجئة — والشبكات الالتفافية
  في الأسبوع القادم هي ما تحصل عليه حين تكفّ عن رمي تلك البنية.
- **والتصنيف رقمُ فئة.** فـ `y` يحمل أعدادًا صحيحة من ٠ إلى ٩ لا متّجهات one-hot، ويأخذها
  `nn.CrossEntropyLoss` كما هي.

وثلاثة تقسيمات، ولكلٍّ منها عمل واحد:

- **التدريب: ١٬٠٠٠ صورة** مسحوبة من مجموعة التدريب ذات الستين ألفًا. وهي قليلة عن قصد. فشبكة بـ
  ٢٧٠٬٠٠٠ معامل تحفظ ألف صورة في حقب معدودة، وفرط المطابقة هو ما يجب أن يكون أمام هذا المعمل. وتُبقي
  أيضًا كل تشغيل في ثوانٍ على معالج حاسوب محمول.
- **التحقّق: ٥٬٠٠٠ صورة تدريب أخرى.** وكل قرار اليوم — Dropout والمُحسِّن ومعدّل التعلّم ومتى تتوقّف —
  يُتّخذ عليها.
- **الاختبار: العشرة آلاف الرسمية.** تُمسّ مرة واحدة في آخر المهمة ٢٫٦. فالرقم الذي ضبطت عليه ليس نتيجة
  اختبار.

وتُحمّل أيضًا **`tree_baseline.json`** من الاثنين للمقارنة. فإن لم تُكمل ذلك المعمل رجعت `load_artefact`
إلى النسخة المرجعية وطبعت ملاحظة.

**انتبه:** MNIST سهلة على نحو غير واقعي — فالشبكة الالتفافية تتجاوز ٩٩٪ عليها، والنتيجة شبه الكاملة هنا
لا تدلّ على شيء في الرؤية الحاسوبية الحقيقية. وبألف صورة تدريب فقط تقع الشبكة متعدّدة الطبقات قرب ٩٠٪،
وهذا هو الرقم الصادق لهذا الإعداد. فلا تقارنه بلوحة صدارة درّبت على الستين ألفًا كلها.

</div>


## Setup

Run the cell below first. It downloads MNIST (first run only), draws the three splits, and runs every
image through the transform **once**, stacking each split into a single tensor.

That last step is a choice worth knowing about. A `torchvision` dataset applies its transform every
time an image is read, and today you will read the same 1,000 images tens of thousands of times. When
a dataset fits in memory — and 16,000 small images is about 50 MB — transforming once and training
from a `TensorDataset` is the standard move. When it does not fit, the `DataLoader` reads and
transforms from disk as it goes, which is what week 4 does with real photographs.

<div dir="rtl" align="right">

## الإعداد

شغّل الخليّة أدناه أولًا. تُنزّل MNIST (في أول تشغيل فقط)، وتسحب التقسيمات الثلاثة، وتمرّر كل صورة في
التحويل **مرة واحدة**، وترصّ كل تقسيم في تنسور واحد.

والخطوة الأخيرة اختيار يستحقّ المعرفة. فمجموعة `torchvision` تطبّق تحويلها في كل مرة تُقرأ فيها صورة،
وستقرأ اليوم الصور الألف نفسها عشرات آلاف المرات. وحين تتّسع الذاكرة للبيانات — و١٦٬٠٠٠ صورة صغيرة
نحو ٥٠ ميغابايت — فالتحويل مرة واحدة ثم التدريب من `TensorDataset` هو الخطوة المعيارية. وحين لا تتّسع،
يقرأ `DataLoader` من القرص ويحوّل وهو يمضي، وهذا ما يفعله الأسبوع الرابع بالصور الحقيقية.

</div>


In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import describe_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("torch", "torchvision", "scikit-learn", "matplotlib", "pyarrow")
seed_everything(42)

import copy
import json
import math

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms

from sklearn.ensemble import HistGradientBoostingClassifier

from aiep.viz import use_course_style, PALETTE, plot_grid
use_course_style()

SEED = 42
N_TRAIN, N_VAL = 1_000, 5_000
MNIST_ROOT = ARTEFACT_DIR / "torchvision"
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

# ToTensor:  a 28x28 image of 0-255 integers -> a (1, 28, 28) float tensor in [0, 1].
# Normalize: subtract MNIST's mean pixel and divide by its std, so inputs centre on 0.
TRANSFORM = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))])
mnist_train = datasets.MNIST(MNIST_ROOT, train=True, download=True, transform=TRANSFORM)
mnist_test = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=TRANSFORM)


def stack(dataset, indices):
    """Run the transform once per image; return (images, labels) as two tensors."""
    loader = DataLoader(Subset(dataset, indices), batch_size=1_000)
    images, labels = zip(*loader)
    return torch.cat(images), torch.cat(labels)


order = torch.randperm(len(mnist_train), generator=torch.Generator().manual_seed(SEED))
X_train, y_train = stack(mnist_train, order[:N_TRAIN].tolist())
X_val, y_val = stack(mnist_train, order[N_TRAIN:N_TRAIN + N_VAL].tolist())
X_test, y_test = stack(mnist_test, list(range(len(mnist_test))))
POOL = order[N_TRAIN + N_VAL:]   # the other 54,000 training images, for task 2.6's fresh draws

tree_baseline = json.loads(load_artefact("tree_baseline.json").read_text(encoding="utf-8"))

print(describe_dataset("mnist"))
print(f"\ntrain {tuple(X_train.shape)} | val {tuple(X_val.shape)} | "
      f"test {tuple(X_test.shape)} | {X_train.dtype}")
print(f"labels are class indices, {y_train.dtype}: {y_train[:12].tolist()}")
print(f"\nMonday's baseline, on credit_default: {tree_baseline['best_model']}, "
      f"AUC {tree_baseline['cv_mean']:.4f} +/- {tree_baseline['cv_std']:.4f}")
print("\n", versions())

## Section 1 — Warm-up: look at the pixels, then diagnose four runs  (≈25 min)

Everything in this section already works.

First the data, because this is the first time this week it is not a table. The cell below shows you
the same digit at every stage — the image on disk, the tensor `ToTensor` makes of it, the normalised
version the network sees — then one batch as a `DataLoader` hands it over, and what `nn.Flatten` does
to that batch. Read the shapes; you will be reading shapes like these for the rest of the bootcamp.

<div dir="rtl" align="right">

## القسم الأول — التهيئة: انظر في البكسلات، ثم شخِّص أربعة تشغيلات (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا.

البيانات أولًا، لأنها أول مرة هذا الأسبوع لا تكون فيها جدولًا. تُريك الخليّة أدناه الرقم نفسه في كل
مرحلة — الصورة على القرص، والتنسور الذي يصنعه منها `ToTensor`، والنسخة المُطبَّعة التي تراها الشبكة — ثم
دفعة واحدة كما يناولها `DataLoader`، وما يفعله `nn.Flatten` بتلك الدفعة. اقرأ الأشكال؛ فستقرأ أشكالًا
كهذه بقية المعسكر.

</div>


In [ ]:
plot_grid(X_train[:10] * MNIST_STD + MNIST_MEAN, y_train[:10].tolist(), n_cols=10,
          title="The first 10 training images, with their labels")
plt.show()

raw = datasets.MNIST(MNIST_ROOT, train=True)          # the same files, with no transform
image, label = raw[int(order[0])]
as_tensor = transforms.ToTensor()(image)
darkest, brightest = image.getextrema()
print(f"on disk:          a {image.mode!r} image of size {image.size}, "
      f"pixels {darkest} to {brightest}, label {label}")
print(f"after ToTensor:   {tuple(as_tensor.shape)} {as_tensor.dtype}, "
      f"pixels {as_tensor.min():.1f} to {as_tensor.max():.1f}")
print(f"after Normalize:  all 1,000 training images have mean {X_train.mean():.3f} "
      f"and std {X_train.std():.3f}")

xb, yb = next(iter(DataLoader(TensorDataset(X_train, y_train), batch_size=32)))
print(f"\none batch:        images {tuple(xb.shape)} = (batch, channels, height, width), "
      f"labels {tuple(yb.shape)}")
print(f"after nn.Flatten: {tuple(nn.Flatten()(xb).shape)}   <- what an MLP sees: 784 numbers, "
      f"no picture")
print(f"\nimages per digit in the training 1,000: {torch.bincount(y_train).tolist()}")

### Task 1.1 — diagnose four runs

The cell below reproduces the four curves from this morning — A, B, C and D — as real training runs
on these 1,000 images, with a pinned seed, not drawings. Each one changed exactly one thing.

Plot them, then write your diagnosis for each in the markdown cell that follows, **before** you run
the answer key in the cell after that. Writing it down first is the exercise; reading the answer with
your own guess already on the page is what makes it stick.

<div dir="rtl" align="right">

### المهمة ١٫١ — شخِّص أربعة تشغيلات

تُعيد الخليّة أدناه إنتاج منحنيات الصباح الأربعة — أ وب وج ود — كتشغيلات تدريب حقيقية على هذه الصور
الألف ببذرة ثابتة لا كرسوم. وكل واحد منها غيّر شيئًا واحدًا بالضبط.

ارسمها، ثم اكتب تشخيصك لكل واحد في خليّة النصّ التالية **قبل** أن تشغّل مفتاح الإجابات في الخليّة
بعدها. فالكتابة أولًا هي التمرين، وقراءة الجواب وتخمينك مكتوب على الورقة هي ما يُثبّته.

</div>


In [ ]:
def small_mlp(width=64, seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, width), nn.ReLU(), nn.Linear(width, 10))


def losses_now(model, criterion):
    model.eval()
    with torch.no_grad():
        return (criterion(model(X_train), y_train).item(),
                criterion(model(X_val), y_val).item())


def full_batch_run(lr, epochs, dead=False, seed=0):
    """One gradient step per epoch on all 1,000 images, so a bad learning rate shows at once."""
    model = small_mlp(seed=seed)
    if dead:   # a first-layer bias this negative silences every ReLU, for every image
        with torch.no_grad():
            model[1].bias.fill_(-50.0)
    optimiser = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    train, val = [], []
    for _ in range(epochs):
        model.train()
        optimiser.zero_grad()
        criterion(model(X_train), y_train).backward()
        optimiser.step()
        t, v = losses_now(model, criterion)
        train.append(t)
        val.append(v)
    return train, val


def mini_batch_run(epochs=60, seed=0):
    """A wide network on 1,000 images, trained a long time."""
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                          nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train, val = [], []
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimiser.zero_grad()
            criterion(model(xb), yb).backward()
            optimiser.step()
        t, v = losses_now(model, criterion)
        train.append(t)
        val.append(v)
    return train, val


CURVES = {
    "A": full_batch_run(lr=5.0, epochs=12),
    "B": full_batch_run(lr=1e-4, epochs=200),
    "C": mini_batch_run(),
    "D": full_batch_run(lr=0.1, epochs=60, dead=True),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (name, (train, val)) in zip(axes.ravel(), CURVES.items()):
    ax.plot(train, color=PALETTE[0], linewidth=2, label="training")
    ax.plot(val, color=PALETTE[1], linewidth=2, label="validation")
    if name == "A":
        ax.set_yscale("log")
    ax.set_title(f"curve {name}:  {train[0]:.4f} -> {train[-1]:.4f} "
                 f"over {len(train)} epochs")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
plt.tight_layout()
plt.show()

for name, (train, val) in CURVES.items():
    print(f"curve {name}: train {train[0]:.4f} -> {train[-1]:.4f} | "
          f"validation {val[0]:.4f} -> {val[-1]:.4f} | best validation {min(val):.4f} "
          f"at epoch {val.index(min(val))}")

**Your four diagnoses**, one line each — what is wrong, and what would you change?

- **A:**
- **B:**
- **C:**
- **D:**

<div dir="rtl" align="right">

**تشخيصاتك الأربعة**، سطر لكل واحد — ما العطل، وما الذي تغيّره؟

- **أ:**
- **ب:**
- **ج:**
- **د:**

</div>


In [ ]:
ANSWER_KEY = {
    "A": ("the learning rate is far too high. Each step overshoots and lands further up the "
          "other side, so the next gradient is larger and the next step is worse — the loss "
          "jumps by orders of magnitude. Fix: divide the learning rate by 10, and again if it "
          "still diverges."),
    "B": ("the learning rate is far too low — 200 epochs moved the loss by a few hundredths. "
          "Fix: raise it 10x. But check the other cause first: if every ReLU output is zero, "
          "the units are dead and no learning rate will help. Which is curve D."),
    "C": ("overfitting. Training loss goes to ~0 while validation bottoms out within a few "
          "epochs and then climbs. Fix: stop at the best epoch, get more data, or regularise "
          "— and measure which of those actually helped rather than assuming."),
    "D": ("nothing at all. 2.3026 is ln 10, which is exactly the loss of a model that gives "
          "every digit probability 1/10 on a balanced ten-class problem. It has not learned a "
          "weak rule; it has learned no rule. Here a first-layer bias of -50 killed every ReLU, "
          "so no gradient could reach the first layer. Look at the activations, the data and "
          "the labels before you touch a hyperparameter."),
}

for name, answer in ANSWER_KEY.items():
    print(f"curve {name}: {answer}\n")

print(f"the numbers to memorise: ln 10 = {math.log(10):.4f} for ten balanced classes,")
print(f"ln 2 = {math.log(2):.4f} for two — in general ln k for k balanced classes")

## Section 2 — Core: six tasks  (≈60 min)

1. Reproduce curve C on purpose, and own the training function every later task calls.
2. Add dropout, and measure whether it helped. Report what you measure, not what you expect.
3. Four optimisers, identical seeds, one figure and a table.
4. Early stopping that restores the best weights.
5. `experiments.parquet` — nine rows, changing one thing at a time.
6. **The reckoning:** your best network against a boosted tree, on the same pixels, on the test set.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. أعِد إنتاج المنحنى ج عن قصد، وامتلك دالة التدريب التي تناديها كل مهمة بعدها.
٢. أضِف Dropout وقِس هل نفع. واعرض ما تقيسه لا ما تتوقّعه.
٣. أربعة مُحسِّنات ببذور متطابقة، وشكل واحد وجدول.
٤. إيقاف مبكر يستعيد أفضل الأوزان.
٥. `experiments.parquet` — تسعة صفوف بتغيير شيء واحد في كل مرة.
٦. **الحساب:** أفضل شبكة عندك مقابل شجرة معزَّزة، على البكسلات نفسها، على مجموعة الاختبار.

</div>


### Task 2.1 — build the overfitting machine

Take a network far larger than 1,000 images need — `nn.Flatten`, then 256 hidden units twice, then 10
outputs, about 270,000 parameters — and train it for 100 epochs. It has enough capacity to memorise
those images many times over, and it will.

Ten classes change two things from Thursday's network, and nothing else:

- **The last layer has 10 outputs** — one logit per digit. The prediction is the largest one:
  `logits.argmax(dim=1)`.
- **The loss is `nn.CrossEntropyLoss`.** It takes the raw logits and the integer labels, and it is
  `BCEWithLogitsLoss`'s ten-class sibling: the softmax and the logarithm fused into one step, so no
  probability is ever formed that could round to 0. No activation on the final layer — Thursday's
  stretch task, now the rule.

Watch for the shape from curve C: the training loss walks down towards zero, the validation loss
bottoms out early and then turns and climbs. **The turn is the moment the model stopped learning
digits and started learning these particular 1,000 images.**

Write the training function once, with every knob as an argument. Every task after this one calls it
with one argument changed, which is what makes task 2.5's experiment log honest by construction
rather than by discipline.

<div dir="rtl" align="right">

### المهمة ٢٫١ — ابنِ آلة فرط المطابقة

خُذ شبكة أكبر بكثير مما تحتاجه ألف صورة — `nn.Flatten` ثم ٢٥٦ وحدة مخفية مرتين ثم ١٠ مُخرجات، أي نحو
٢٧٠٬٠٠٠ معامل — ودرّبها مئة حقبة. فسعتها تكفي لحفظ تلك الصور مرات كثيرة، وستحفظها.

والفئات العشر تغيّر أمرين عن شبكة الخميس، ولا شيء غيرهما:

- **للطبقة الأخيرة ١٠ مُخرجات** — logit لكل رقم. والتنبّؤ هو أكبرها: `logits.argmax(dim=1)`.
- **والخسارة `nn.CrossEntropyLoss`.** تأخذ الـ logits الخام والتصنيفات الصحيحة، وهي أخت
  `BCEWithLogitsLoss` ذات الفئات العشر: الـ softmax واللوغاريتم مدموجان في خطوة واحدة، فلا يتكوّن احتمال
  قد يُقرَّب إلى صفر. ولا تنشيط على الطبقة الأخيرة — مهمة الخميس الإضافية وقد صارت القاعدة.

وراقب الشكل من المنحنى ج: تمشي خسارة التدريب نازلة نحو الصفر، وتبلغ خسارة التحقّق قاعها مبكرًا ثم تستدير
وتصعد. **والاستدارة هي اللحظة التي توقّف فيها النموذج عن تعلّم الأرقام وبدأ يحفظ هذه الصور الألف بعينها.**

اكتب دالة التدريب مرة واحدة وكل مقبض فيها وسيط. وكل مهمة بعدها تناديها بوسيط واحد مُغيَّر، وهذا ما
يجعل سجلّ تجارب المهمة ٢٫٥ صادقًا بحكم البناء لا بحكم الانضباط.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) build_model(width, dropout, batch_norm, seed) returns an nn.Sequential that
#    starts with nn.Flatten() and ends with nn.Linear(width, 10). Seed it with
#    torch.manual_seed inside, so two calls with the same seed give the same
#    initial weights — every comparison today depends on that.
# 2) train(...) takes the model's settings AND the run's settings (optimiser,
#    learning rate, batch size, epochs) and returns a DataFrame of per-epoch
#    train loss, validation loss and validation accuracy.
# 3) nn.CrossEntropyLoss on raw logits, integer labels, no final activation.
#    Accuracy is (logits.argmax(dim=1) == y).float().mean().
# 4) model.train() inside the epoch, model.eval() and torch.no_grad() to score it.
# Search: "pytorch CrossEntropyLoss MNIST MLP train validation loop"
# https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
#
# ١) `build_model(width, dropout, batch_norm, seed)` تُرجع `nn.Sequential` يبدأ بـ
#    `nn.Flatten()` وينتهي بـ `nn.Linear(width, 10)`. وابذُرها بـ `torch.manual_seed`
#    داخلها، فيُعطي نداءان بالبذرة نفسها الأوزان الابتدائية نفسها — وكل مقارنة
#    اليوم تعتمد على ذلك.
# ٢) تأخذ `train(...)` إعدادات النموذج **و**إعدادات التشغيل (المُحسِّن ومعدّل
#    التعلّم وحجم الدفعة وعدد الحقب) وتُرجع `DataFrame` فيه لكل حقبة خسارة التدريب
#    وخسارة التحقّق ودقة التحقّق.
# ٣) `nn.CrossEntropyLoss` على الـ logits الخام والتصنيفات الصحيحة، بلا تنشيط أخير.
#    والدقة هي `(logits.argmax(dim=1) == y).float().mean()`.
# ٤) `model.train()` داخل الحقبة، و`model.eval()` مع `torch.no_grad()` للتقييم.
# ابحث عن: "pytorch CrossEntropyLoss MNIST MLP train validation loop"
# ────────────────────────────────────────────────────────────────────

# TODO: Write build_model(width, dropout, batch_norm, seed) -> nn.Sequential.
# مهمة: اكتب `build_model(width, dropout, batch_norm, seed)` ← `nn.Sequential`.
# TODO: Write train(...) returning a per-epoch DataFrame, the fitted model and the best epoch's weights.
# مهمة: اكتب `train(...)` تُرجع `DataFrame` لكل حقبة، والنموذج المُدرَّب، وأوزان أفضل حقبة.
# TODO: Run it: width 256, no dropout, 100 epochs. This is curve C, on your own code.
# مهمة: شغّلها: عرض ٢٥٦ وبلا Dropout و١٠٠ حقبة. وهذا هو المنحنى ج بشيفرتك أنت.
n_weights = sum(p.numel() for p in overfit_model.parameters())
best_epoch = int(overfit_history["val_loss"].idxmin())
print(f"{n_weights:,} parameters for {len(X_train):,} training images")
print(f"train loss:      {overfit_history.iloc[0]['train_loss']:.4f} -> "
      f"{overfit_history.iloc[-1]['train_loss']:.4f}")
print(f"validation loss: {overfit_history.iloc[0]['val_loss']:.4f} -> "
      f"{overfit_history.iloc[-1]['val_loss']:.4f}")
print(f"best validation: {overfit_history['val_loss'].min():.4f} at epoch "
      f"{best_epoch}, and then it climbed for {len(overfit_history) - 1 - best_epoch} "
      f"more epochs")
print(f"validation accuracy at the end: {overfit_history.iloc[-1]['val_accuracy']:.3f}")

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Both losses against the epoch, one panel. Mark the best validation epoch
#    with a star — that is the model you would have wanted to keep.
# 2) Shade or annotate the region after it, where training continued and the
#    model got worse at the only thing that matters.
# Search: "matplotlib axvline annotate best epoch"
# https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axvline.html
#
# ١) الخسارتان مقابل الحقبة في لوحة واحدة. وأشِر بنجمة إلى حقبة أفضل تحقّق — فهي
#    النموذج الذي كنت تودّ الاحتفاظ به.
# ٢) ظلّل أو علّق على المنطقة بعدها، حيث تواصل التدريب وساء النموذج في الشيء
#    الوحيد الذي يهمّ.
# ابحث عن: "matplotlib axvline annotate best epoch"
# ────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 4.5))
# TODO: Plot both curves and mark the best validation epoch.
# مهمة: ارسم المنحنيين وأشِر إلى حقبة أفضل تحقّق.
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy")
ax.set_title("Curve C, reproduced on purpose")
ax.legend()
plt.show()

### Task 2.2 — add dropout, and measure it honestly

`nn.Dropout(0.5)` zeroes half of a layer's activations on every forward pass, chosen fresh each time.
No path through the network can be relied upon, so no unit is allowed to become indispensable.

Retrain the same network with `dropout=0.5` — one argument changed, everything else identical,
including the seed.

**Two numbers, and they do not move together.** The training loss should get **worse**: you have made
the network's job harder on purpose, and during training half its units are silenced at random.
Whether the validation loss gets **better** is the actual experiment, and it is not guaranteed. The
lecture was explicit that dropout is a real tool and not a reliable win: it earns its reputation on
wide, redundant inputs, and it can hurt on a handful of columns.

784 pixels, most of them saying the same thing as their neighbours, is exactly the redundant kind of
input. So this is the fair test. Record both numbers and write down which way it went. If it made
things worse, say so — that is a result, and a lab that only reports the runs that worked is not a lab.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — أضِف Dropout وقِسه بصدق

يُصفّر `nn.Dropout(0.5)` نصف تنشيطات الطبقة في كل مرور أمامي، ويُختارون من جديد كل مرة. فلا يمكن
الاعتماد على أي مسار في الشبكة، فلا يُسمح لأي وحدة أن تصير لا غنى عنها.

أعِد تدريب الشبكة نفسها بـ `dropout=0.5` — وسيط واحد مُغيَّر وكل ما عداه متطابق، والبذرة كذلك.

**ورقمان لا يتحرّكان معًا.** يُتوقّع أن **تسوء** خسارة التدريب: فقد صعّبت عمل الشبكة عن قصد، ونصف
وحداتها تُسكَت عشوائيًا أثناء التدريب. أما هل **تتحسّن** خسارة التحقّق فهي التجربة الفعلية، وليست
مضمونة. وقد كانت المحاضرة صريحة في أن Dropout أداة حقيقية لا فوز موثوق: فهو يكتسب سمعته على المدخلات
العريضة الفائضة، وقد يضرّ على حفنة من الأعمدة.

و٧٨٤ بكسلًا يقول أكثرها ما يقوله جيرانه هو بالضبط النوع الفائض من المدخلات. فهذا هو الاختبار المُنصف.
فسجّل الرقمين واكتب في أي اتجاه ذهب. وإن أساء فقُل ذلك — فهذه نتيجة، والمعمل الذي لا يعرض إلا
التشغيلات الناجحة ليس معملًا.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) One call to train(), with dropout=0.5 and everything else exactly as before.
# 2) Compare four numbers: final train loss, final validation loss, best
#    validation loss, and the epoch each best happened at.
# 3) Put them in a small DataFrame — you will paste this shape into
#    experiments.parquet in task 2.5.
# Search: "pytorch nn.Dropout training vs eval behaviour"
# https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html
#
# ١) نداء واحد لـ `train()` بـ `dropout=0.5` وكل ما عداه كما كان بالضبط.
# ٢) قارن أربعة أرقام: خسارة التدريب النهائية، وخسارة التحقّق النهائية، وأفضل
#    خسارة تحقّق، والحقبة التي حدث فيها كل أفضل.
# ٣) ضعها في `DataFrame` صغير — فستنسخ هذا الشكل إلى `experiments.parquet` في
#    المهمة ٢٫٥.
# ابحث عن: "pytorch nn.Dropout training vs eval behaviour"
# ────────────────────────────────────────────────────────────────────

# TODO: Retrain with dropout=0.5, changing nothing else.
# مهمة: أعِد التدريب بـ `dropout=0.5` بلا تغيير أي شيء آخر.
comparison = pd.DataFrame([
    {"run": "no dropout",
     "final_train": overfit_history.iloc[-1]["train_loss"],
     "final_val": overfit_history.iloc[-1]["val_loss"],
     "best_val": overfit_history["val_loss"].min(),
     "best_epoch": int(overfit_history["val_loss"].idxmin()),
     "best_val_accuracy": overfit_history["val_accuracy"].max()},
    {"run": "dropout 0.5",
     "final_train": dropout_history.iloc[-1]["train_loss"],
     "final_val": dropout_history.iloc[-1]["val_loss"],
     "best_val": dropout_history["val_loss"].min(),
     "best_epoch": int(dropout_history["val_loss"].idxmin()),
     "best_val_accuracy": dropout_history["val_accuracy"].max()},
])
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
train_worse = (comparison.iloc[1]["final_train"] > comparison.iloc[0]["final_train"])
val_better = (comparison.iloc[1]["best_val"] < comparison.iloc[0]["best_val"])
print(f"\ndropout made the training loss worse: {train_worse}   "
      f"<- this is the regularisation working")
print(f"dropout made the best validation loss better: {val_better}   "
      f"<- this is the question")

On pixels, dropout helped — and the way it helped is worth reading closely.

The best validation loss went from about **0.40 to 0.36**, and the epoch it happened at moved from
**2 to 14**. That second number is the real effect: dropout did not stop the model memorising, it
slowed it down, so the good model existed for longer and ended up better. The training loss, as
predicted, was worse at every epoch.

Now look at the *final* validation loss. It is still far above the best one — worse, in fact, than the
run without dropout ended. Train long enough and the network memorises 1,000 images through dropout
too. **So dropout is not a substitute for stopping at the right epoch; it moves the right epoch later
and makes it better.** Task 2.4 is about stopping there.

On Thursday's `moons_toy` — two input columns, nothing redundant to remove — the same layer made both
losses worse. Same code, opposite result, and the difference is the input. That is the reason you
measure it every time rather than switching it on by habit.

<div dir="rtl" align="right">

على البكسلات نفع Dropout — والطريقة التي نفع بها تستحقّ القراءة المتأنّية.

فقد تحرّكت أفضل خسارة تحقّق من نحو **٠٫٤٠ إلى ٠٫٣٦**، وانتقلت الحقبة التي حدثت فيها من **٢ إلى ١٤**.
وهذا الرقم الثاني هو الأثر الحقيقي: فلم يمنع Dropout النموذج من الحفظ بل أبطأه، فبقي النموذج الجيد
موجودًا مدة أطول وانتهى أفضل. وخسارة التدريب، كما تُوقِّع، كانت أسوأ في كل حقبة.

وانظر الآن في خسارة التحقّق **النهائية**. فهي ما زالت أعلى بكثير من أفضلها — بل أسوأ مما انتهى إليه
التشغيل بلا Dropout. فإن درّبت مدة كافية حفظت الشبكة ألف صورة عبر Dropout أيضًا. **فـ Dropout ليس بديلًا
عن التوقّف في الحقبة الصحيحة؛ بل يؤخّر الحقبة الصحيحة ويجعلها أفضل.** والمهمة ٢٫٤ عن التوقّف هناك.

وعلى `moons_toy` يوم الخميس — عمودا مُدخلات ولا فائض يُزال — أساءت الطبقة نفسها الخسارتين. الشيفرة
نفسها والنتيجة عكسها، والفرق هو المُدخل. ولهذا تقيسه في كل مرة بدل أن تشغّله بحكم العادة.

</div>


### Task 2.3 — four optimisers, one seed

SGD, SGD with momentum, RMSprop, Adam. One line changes between them; everything else — the
architecture, the initial weights, the batch order, the number of epochs — is identical.

**Prove that it is identical**, do not assert it. If all four runs start from the same initial loss
to six decimal places, they started from the same weights on the same data. If they do not, your
comparison is measuring initialisation and you would not know.

Then the table: epochs to reach a validation loss of 0.45, and the best validation loss inside the
budget. Plain SGD gets there last. That is the point of including it.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — أربعة مُحسِّنات وبذرة واحدة

SGD، وSGD مع الزخم، وRMSprop، وAdam. يتغيّر سطر واحد بينها، وكل ما عداه — المعمارية والأوزان
الابتدائية وترتيب الدفعات وعدد الحقب — متطابق.

**وأثبِت أنه متطابق** ولا تدّعِه. فإن بدأت التشغيلات الأربعة من الخسارة الابتدائية نفسها إلى ست خانات
عشرية فقد بدأت من الأوزان نفسها على البيانات نفسها. وإن لم تبدأ فمقارنتك تقيس التهيئة وأنت لا تدري.

ثم الجدول: عدد الحقب للوصول إلى خسارة تحقّق ٠٫٤٥، وأفضل خسارة تحقّق داخل الميزانية. ويصل SGD
العادي آخرها، وهذا هو سبب إدراجه.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Loop over the four optimiser names, calling train() with 50 epochs and
#    everything else at its default. One argument changes.
# 2) Keep each history. For the table: the first epoch whose validation loss is
#    below 0.45 (or "never"), and the best validation loss reached.
# 3) Plot all four validation curves on one figure, and check the loss BEFORE
#    the first step matches — round to six decimals and compare. The loss after
#    epoch 1 already differs; that is the optimiser, not the initialisation.
# Search: "torch.optim SGD momentum RMSprop Adam comparison"
# https://pytorch.org/docs/stable/optim.html
#
# ١) كرّر على أسماء المُحسِّنات الأربعة مناديًا `train()` بخمسين حقبة وكل ما عداه على
#    قيمته الافتراضية. ويتغيّر وسيط واحد.
# ٢) احتفظ بكل سجلّ. وللجدول: أول حقبة تكون خسارة تحقّقها دون ٠٫٤٥ (أو «لم يصل»)،
#    وأفضل خسارة تحقّق بلغها.
# ٣) ارسم منحنيات التحقّق الأربعة في شكل واحد، وتحقّق من تطابق الخسارة **قبل**
#    الخطوة الأولى بتقريبها إلى ست خانات عشرية ومقارنتها. أما الخسارة بعد الحقبة
#    الأولى فتختلف أصلًا، وذلك من المُحسِّن لا من التهيئة.
# ابحث عن: "torch.optim SGD momentum RMSprop Adam comparison"
# ────────────────────────────────────────────────────────────────────

OPTIMISERS = ["sgd", "momentum", "rmsprop", "adam"]
TARGET_LOSS = 0.45
# TODO: Train once per optimiser, 50 epochs, everything else identical.
# مهمة: درّب مرة لكل مُحسِّن، خمسين حقبة، وكل ما عداه متطابق.
# TODO: Build the table: epochs to reach TARGET_LOSS, and the best validation loss.
# مهمة: ابنِ الجدول: الحقب اللازمة لبلوغ `TARGET_LOSS`، وأفضل خسارة تحقّق.
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, colour in zip(OPTIMISERS, PALETTE):
    ax.plot(optimiser_runs[name]["val_loss"], color=colour, linewidth=2, label=name)
ax.axhline(TARGET_LOSS, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("epoch")
ax.set_ylabel("validation loss")
ax.set_title("One line changed, four different trainings")
ax.legend()
plt.show()
print(optimiser_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
same_start = optimiser_table["loss_before_training"].round(6).nunique() == 1
print(f"\nall four started from the same weights: {same_start}   "
      f"(training loss before the first step, to 6 decimals — about ln 10)")

### Task 2.4 — early stopping, and the number it is worth

Task 2.1 showed the model you wanted appearing early and then being trained past. Early stopping is
the fix, and it has two halves — only one of which people remember.

**Stop when validation stops improving.** `patience=10` means: after 10 epochs with no improvement,
give up.

**Restore the best weights, not the last ones.** These are different models. `model.load_state_dict(best_state)`
is the line people forget, and forgetting it turns early stopping into "training for slightly fewer
epochs", which is worth almost nothing.

Run with `patience=10`, then score the model twice: as it was at the final epoch, and with the best
weights restored. Report both.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الإيقاف المبكر والرقم الذي يساويه

أرَتك المهمة ٢٫١ النموذج الذي أردته يظهر مبكرًا ثم يُتجاوَز بالتدريب. والإيقاف المبكر هو العلاج، وله
نصفان لا يتذكّر الناس إلا واحدًا منهما.

**توقّف حين تتوقّف نتيجة التحقّق عن التحسّن.** فـ `patience=10` تعني: بعد عشر حقب بلا تحسّن، استسلم.

**واستعِد أفضل الأوزان لا آخرها.** فهذان نموذجان مختلفان. و`model.load_state_dict(best_state)` هو
السطر الذي ينساه الناس، ونسيانه يحوّل الإيقاف المبكر إلى «تدريب بحقب أقل قليلًا»، وهذا لا يساوي شيئًا
تقريبًا.

شغّل بـ `patience=10`، ثم قيّم النموذج مرتين: كما كان في الحقبة الأخيرة، وبأفضل الأوزان مستعادةً.
واعرض الرقمين.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Call train() with patience=10 — it already returns best_state alongside the
#    model. Note how many epochs it actually ran for.
# 2) Score the model as it stands (the final epoch's weights) on the validation
#    set, under eval() and no_grad().
# 3) Then load_state_dict(best_state) and score it again. Two numbers, one model
#    object, and the difference is what the line is worth.
# Search: "pytorch early stopping restore best weights state_dict"
# https://pytorch.org/tutorials/beginner/saving_loading_models.html
#
# ١) نادِ `train()` بـ `patience=10` — فهي تُرجع `best_state` مع النموذج أصلًا.
#    ولاحظ كم حقبة عملت فعلًا.
# ٢) قيّم النموذج كما هو (أوزان الحقبة الأخيرة) على مجموعة التحقّق داخل `eval()`
#    و`no_grad()`.
# ٣) ثم `load_state_dict(best_state)` وقيّمه مرة أخرى. رقمان وكائن نموذج واحد،
#    والفرق بينهما هو ما يساويه ذلك السطر.
# ابحث عن: "pytorch early stopping restore best weights state_dict"
# ────────────────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()
def score(model, X, y):
    """Loss and accuracy, in eval mode, without building a graph."""
    model.eval()
    with torch.no_grad():
        logits = model(X)
        return (criterion(logits, y).item(),
                (logits.argmax(dim=1) == y).float().mean().item())
# TODO: Train with patience=10, then score the final weights and the restored best ones.
# مهمة: درّب بـ `patience=10`، ثم قيّم الأوزان النهائية ثم أفضل الأوزان المستعادة.
print(f"stopped after {len(stopped_history)} epochs of a 100-epoch budget")
print(f"  last epoch's weights:  val loss {final_loss:.4f}, "
      f"accuracy {final_accuracy:.3f}")
print(f"  best epoch's weights:  val loss {restored_loss:.4f}, "
      f"accuracy {restored_accuracy:.3f}")
print(f"\nrestoring the best weights was worth "
      f"{final_loss - restored_loss:.4f} of validation loss")
print(f"and against the 100-epoch run from task 2.1 "
      f"({overfit_history.iloc[-1]['val_loss']:.4f}), it is worth "
      f"{overfit_history.iloc[-1]['val_loss'] - restored_loss:.4f}")

Put the last line next to task 2.2 and the ranking is clear, and a little surprising.

Early stopping added no hyperparameter to the model at all. It simply declined to keep training, and
it recovered most of what 100 epochs of memorisation had thrown away — while also finishing in a
fraction of the time. **The best model already happened; training continued past it.**

Dropout, on these pixels, did something different and also useful: it made the best model better and
made it happen later. The two are not rivals. Dropout decides how good the best epoch can be; early
stopping decides whether you keep it. Task 2.5 will tell you which one mattered more here.

The general lesson survives outside this dataset: before reaching for a regulariser, check whether you
are simply keeping the wrong epoch. It is free, it always applies, and it is the first thing to try.

**One more thing, and do not skip it: look at the accuracies.** The restored weights have the *lower*
loss and, very likely, the *lower* accuracy too. That is Wednesday's lesson again. Early stopping here
watches the validation **loss**, which cares how confident each answer is; accuracy only cares which
digit won. Early in training the model is right about most images but not yet sure of them; later it is
right about slightly more of them and badly, confidently wrong about the rest. Which one you stop on is
a decision, not a default — and if the number you will report is accuracy, you can stop on accuracy.

<div dir="rtl" align="right">

ضَع السطر الأخير بجانب المهمة ٢٫٢ فيصير الترتيب واضحًا، ومفاجئًا قليلًا.

فلم يُضِف الإيقاف المبكر إلى النموذج أي معامل. بل امتنع عن مواصلة التدريب فحسب، فاستعاد أكثر ما رمته
مئة حقبة من الحفظ — وانتهى أيضًا في جزء من الوقت. **فالنموذج الأفضل قد حدث أصلًا، والتدريب واصل بعده.**

أما Dropout فقد فعل على هذه البكسلات شيئًا مختلفًا ونافعًا أيضًا: جعل أفضل نموذج أفضل وأخّر حدوثه. وهما
ليسا متنافسين. فـ Dropout يقرّر كم يمكن أن تكون أفضل حقبة جيدة، والإيقاف المبكر يقرّر هل تحتفظ بها. وستُخبرك
المهمة ٢٫٥ أيّهما كان أهمّ هنا.

والدرس العام يصمد خارج هذه البيانات: قبل أن تمدّ يدك إلى مُنظِّم، تحقّق هل أنت ببساطة تحتفظ بالحقبة
الخطأ. فهذا مجّاني، وينطبق دائمًا، وهو أول ما تُجرّبه.

**وأمر آخر فلا تتخطَّه: انظر في الدقّتين.** فالأوزان المستعادة لها الخسارة **الأقل**، وعلى الأرجح الدقة
**الأقل** أيضًا. وهذا درس الأربعاء مرة أخرى. فالإيقاف المبكر هنا يراقب **خسارة** التحقّق، وهي يهمّها كم
كان كل جواب واثقًا، أما الدقة فلا يهمّها إلا أي رقم فاز. ففي أول التدريب يصيب النموذج أغلب الصور دون أن
يكون متيقّنًا منها، وبعد ذلك يصيب عددًا أكبر قليلًا ويخطئ في الباقي خطأً واثقًا فادحًا. وأيّهما تتوقّف عنده
قرارٌ لا قيمة افتراضية — فإن كان الرقم الذي ستعرضه هو الدقة فلك أن تتوقّف على الدقة.

</div>


### Task 2.5 — the experiment log

Every number you have produced today is currently a print statement that will be gone the moment you
restart the kernel. Now write them down properly.

`experiments.parquet`: one row per run, with **every** hyperparameter, the seed, both losses and the
validation accuracy. At least eight rows, and — this is the rule that matters — **each row differs
from the baseline by exactly one hyperparameter.**

That rule is the transferable skill of the day and it is the one people abandon under time pressure.
Change two things, get a better number, and you have learned nothing: you cannot say which change
helped, and you cannot undo the one that hurt. The check at the end of this notebook walks every row against
the baseline and fails if any of them moved two knobs, because the instruction alone has never been
enough.

This runs nine trainings and takes about half a minute.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — سجلّ التجارب

كل رقم أنتجته اليوم هو الآن جملة طباعة تختفي لحظة إعادة تشغيل النواة. فاكتبها الآن كما ينبغي.

`experiments.parquet`: صف لكل تشغيل فيه **كل** معامل، والبذرة، والخسارتان، ودقة التحقّق. وثمانية صفوف
على الأقل، والقاعدة المهمّة: **يختلف كل صف عن خط الأساس بمعامل واحد بالضبط.**

وهذه القاعدة هي المهارة المنقولة في اليوم، وهي التي يتركها الناس تحت ضغط الوقت. فإن غيّرت شيئين وحصلت
على رقم أفضل فلم تتعلّم شيئًا: إذ لا تستطيع قول أي تغيير نفع، ولا تستطيع التراجع عن الذي ضرّ. ويفرض
الفحص في آخر هذا الدفتر ذلك، لأن التعليمة وحدها لم تكفِ يومًا.

وهذا يشغّل تسعة تدريبات ويستغرق نحو نصف دقيقة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Define one BASE config as a dict. Then define the variants as (name, key,
#    value) — each one copies BASE and overrides a single key.
# 2) Run each, and record the config alongside the results, so the file can be
#    read a month from now without this notebook.
# 3) Include what changed as its own column. A log you have to diff by eye is a
#    log nobody reads.
# Search: "python dict copy override config experiment log pandas"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html
#
# ١) عرّف إعداد `BASE` واحدًا كقاموس. ثم عرّف المتغيّرات كثلاثيّات (اسم، مفتاح،
#    قيمة) — وكل واحدة تنسخ `BASE` وتتجاوز مفتاحًا واحدًا.
# ٢) شغّل كل واحدة وسجّل الإعداد مع النتائج، فيمكن قراءة الملف بعد شهر بلا هذا
#    الدفتر.
# ٣) وأدرِج «ما الذي تغيّر» عمودًا مستقلًا. فالسجلّ الذي تقارنه بعينك
#    لا يقرؤه أحد.
# ابحث عن: "python dict copy override config experiment log pandas"
# ────────────────────────────────────────────────────────────────────

BASE = {"width": 256, "dropout": 0.0, "batch_norm": False, "optimiser": "adam",
        "lr": 1e-3, "batch_size": 32, "epochs": 50, "seed": SEED}
VARIANTS = [("baseline", None, None),
            ("narrower", "width", 32),
            ("dropout 0.5", "dropout", 0.5),
            ("sgd", "optimiser", "sgd"),
            ("momentum", "optimiser", "momentum"),
            ("rmsprop", "optimiser", "rmsprop"),
            ("lr 1e-2", "lr", 1e-2),
            ("lr 1e-4", "lr", 1e-4),
            ("batch 128", "batch_size", 128)]
# TODO: Run every variant and collect one row per run, config included.
# مهمة: شغّل كل متغيّر واجمع صفًا لكل تشغيل مع إعداده.
columns = ["run", "changed", "value", "final_train_loss", "best_val_loss",
           "best_epoch", "final_val_accuracy"]
print(experiments[columns].to_string(index=False,
                                     float_format=lambda v: f"{v:.4f}"))
best_run = experiments.loc[experiments["best_val_loss"].idxmin()]
print(f"\nbest configuration: {best_run['run']} "
      f"(best validation loss {best_run['best_val_loss']:.4f})")
print(f"and it is the best of {len(experiments)} — say that when you report it")
# Pulled back out of the DataFrame, so the numeric types are numpy's. torch wants
# Python ints, and this is the kind of edge that only appears when you reuse a config.
best_config = {key: best_run[key] for key in BASE}
for key in ("width", "batch_size", "epochs", "seed"):
    best_config[key] = int(best_config[key])
best_config["dropout"] = float(best_config["dropout"])
best_config["lr"] = float(best_config["lr"])
best_config["batch_norm"] = bool(best_config["batch_norm"])
print(f"best_config, for task 2.6 and the checkpoint: {best_config}")

### Task 2.6 — the reckoning

Everything until now was tuning. This is the question the week was built around.

On Monday, on a table of 23 named columns, the best model was a boosted tree. Today the same kind of
tree — `HistGradientBoostingClassifier` — gets the pixels. Put your best configuration from task 2.5
next to it and let the test set decide.

Three rules, and they are all about fairness rather than about networks:

- **The same rows, the same view.** Both models train on the same 1,000 images, and both see them as
  784 numbers: the tree gets `X.flatten(1)`, which is exactly what your MLP sees after `nn.Flatten`.
  Neither is told which pixels were neighbours.
- **Five draws, not one.** Draw five different sets of 1,000 training images from the 54,000 you have
  not used, train both models on each, and report mean and spread. One draw against one draw is an
  anecdote.
- **The test set, once.** All ten models are scored on the same 10,000 test images, one time, at the
  end. Nothing is tuned on them. The network early-stops on the validation split, as in task 2.4.

**Report it either way.** Monday the tree won; if it wins again, say so.

This trains ten models and takes about two minutes — the tree is the slow one.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الحساب

كل ما سبق ضبط. وهذا هو السؤال الذي بُني الأسبوع حوله.

كان أفضل نموذج يوم الاثنين، على جدول من ٢٣ عمودًا مُسمّى، شجرةً معزَّزة. واليوم يأخذ النوع نفسه من الأشجار —
`HistGradientBoostingClassifier` — البكسلات. فضَع أفضل إعداد عندك من المهمة ٢٫٥ بجانبه ودع مجموعة
الاختبار تحكم.

وثلاث قواعد، وكلها عن الإنصاف لا عن الشبكات:

- **الصفوف نفسها والنظرة نفسها.** يتدرّب النموذجان على الصور الألف نفسها، ويراها كلاهما ٧٨٤ رقمًا: فتأخذ
  الشجرة `X.flatten(1)`، وهو بالضبط ما تراه شبكتك بعد `nn.Flatten`. ولا يُخبَر أيٌّ منهما أي البكسلات
  كانت متجاورة.
- **خمس سحبات لا سحبة واحدة.** اسحب خمس مجموعات مختلفة من ألف صورة تدريب من الـ ٥٤٬٠٠٠ التي لم تستخدمها،
  ودرّب النموذجين على كل واحدة، واعرض المتوسط ومدى التفاوت. فسحبة مقابل سحبة حكاية لا نتيجة.
- **مجموعة الاختبار مرة واحدة.** تُقاس النماذج العشرة كلها على صور الاختبار العشرة آلاف نفسها مرة واحدة
  في النهاية. ولا يُضبط عليها شيء. وتتوقّف الشبكة مبكرًا على تقسيم التحقّق كما في المهمة ٢٫٤.

**وأبلِغ في الحالتين.** فقد فازت الشجرة يوم الاثنين؛ فإن فازت مرة أخرى فقُل ذلك.

وهذا يدرّب عشرة نماذج ويستغرق نحو دقيقتين — والشجرة هي البطيئة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) For each draw d in range(N_DRAWS): take N_TRAIN indices from POOL with
#    torch.randperm and a generator seeded with d, and stack() them into tensors.
# 2) The network: train(**best_config, patience=10, data=(X_draw, y_draw, X_val,
#    y_val)), load its best_state, and score it on X_test with score().
# 3) The tree: HistGradientBoostingClassifier(random_state=SEED).fit on
#    X_draw.flatten(1).numpy() and y_draw.numpy(); .score() it on the test set
#    flattened the same way. scikit-learn wants NumPy arrays, not tensors.
# 4) Keep the last draw's network as last_net — the cell draws its mistakes.
# Search: "sklearn HistGradientBoostingClassifier image pixels flatten"
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html
#
# ١) لكل سحبة `d` في `range(N_DRAWS)`: خُذ `N_TRAIN` موضعًا من `POOL` بـ
#    `torch.randperm` ومولّد مبذور بـ `d`، ثم `stack()` لتحويلها إلى تنسورات.
# ٢) الشبكة: `train(**best_config, patience=10, data=(X_draw, y_draw, X_val,
#    y_val))`، ثم حمّل `best_state` وقِسها على `X_test` بـ `score()`.
# ٣) الشجرة: `HistGradientBoostingClassifier(random_state=SEED).fit` على
#    `X_draw.flatten(1).numpy()` و`y_draw.numpy()`، ثم `.score()` على مجموعة الاختبار
#    مسطّحةً بالطريقة نفسها. فـ scikit-learn يريد مصفوفات NumPy لا تنسورات.
# ٤) احتفظ بشبكة السحبة الأخيرة باسم `last_net` — فالخليّة ترسم أخطاءها.
# ابحث عن: "sklearn HistGradientBoostingClassifier image pixels flatten"
# ────────────────────────────────────────────────────────────────────

N_DRAWS = 5
# TODO: Train the best network and a boosted tree on each of N_DRAWS fresh draws of 1,000 images, score every model once on the test set, and end with two tensors of test accuracies, network_scores and tree_scores, plus the last draw's network as last_net.
# مهمة: درّب أفضل شبكة وشجرة معزَّزة على كل واحدة من `N_DRAWS` سحبة جديدة من ألف صورة، وقِس كل نموذج مرة واحدة على مجموعة الاختبار، وانتهِ بتنسورَي دقة اختبار `network_scores` و`tree_scores`، وبشبكة السحبة الأخيرة باسم `last_net`.
network_mean, network_std = network_scores.mean().item(), network_scores.std().item()
tree_mean, tree_std = tree_scores.mean().item(), tree_scores.std().item()
print(f"\n{'model, 1,000 training images':<36}{'test acc':>10}{'spread':>10}")
print("-" * 56)
print(f"{'MLP (' + best_run['run'] + ')':<36}{network_mean:>10.4f}{network_std:>10.4f}")
print(f"{'HistGradientBoostingClassifier':<36}{tree_mean:>10.4f}{tree_std:>10.4f}")
gap = network_mean - tree_mean
ahead = "the network" if gap > 0 else "the tree"
print(f"\ndifference: {gap:+.4f} of test accuracy — {ahead} is ahead")
overlap = abs(gap) < (network_std + tree_std)
print("and the two spreads "
      + ("overlap — treat this as a tie until you have more draws" if overlap
         else "do not overlap — the difference is larger than the noise"))
print(f"\nfor contrast, Monday on credit_default: {tree_baseline['best_model']} "
      f"won the table, AUC {tree_baseline['cv_mean']:.4f}")
last_net.eval()
with torch.no_grad():
    predicted = last_net(X_test).argmax(dim=1)
mistakes = (predicted != y_test).nonzero().ravel()[:10]
plot_grid(X_test[mistakes] * MNIST_STD + MNIST_MEAN,
          [f"{int(y_test[i])} -> {int(predicted[i])}" for i in mistakes], n_cols=10,
          title="The network's mistakes on the test set (true -> predicted)")
plt.show()

**Write the result down here, in your own words**, in two or three sentences. Name the winner, give
both numbers with their spreads, and say whether the difference is larger than the spreads. Then look
at the grid of mistakes and say what kind of digit the network gets wrong.

Then the sentence that is actually being marked: **Monday a tree won on a table; what happened today,
and what does the difference between the two datasets tell you about when to reach for a network?**

<div dir="rtl" align="right">

**اكتب النتيجة هنا بكلماتك** في جملتين أو ثلاث. سمِّ الفائز، واذكر الرقمين بمدى تفاوت كلٍّ منهما، وقُل هل
الفرق أكبر من مدى التفاوت. ثم انظر في شبكة الأخطاء وقُل أي نوع من الأرقام تخطئ فيه الشبكة.

ثم الجملة التي تُقيَّم فعلًا: **فازت شجرة يوم الاثنين على جدول؛ فماذا حدث اليوم، وماذا يقول لك الفرق بين
المجموعتين عن متى تمدّ يدك إلى شبكة؟**

</div>


**Your report:** _(winner, both numbers, the mistakes, and what the two datasets tell you)_

<div dir="rtl" align="right">

**تقريرك:** _(الفائز، والرقمان، والأخطاء، وما تقوله لك المجموعتان)_

</div>


## Section 3 — Stretch: batch norm, and the batch size that breaks it  (≈30 min)

Open-ended. Lower expectation of completeness.

`nn.BatchNorm1d` normalises each hidden layer's inputs across the batch, so the next layer sees a stable
distribution instead of one that shifts every step. It buys faster training, more tolerance of a
badly chosen learning rate, and a mild regularising effect for free.

Add it, rerun, and record the change. Then set `batch_size=2` and rerun again.

At a batch of 2 you are normalising by the mean and variance of **two images**, which is not a
distribution — it is noise that lurches every step. Below about 8 batch norm stops helping and starts
hurting. Measure it rather than believing it, then write the one sentence explaining why, and answer
the closing question in the cell after.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: تطبيع الدفعات وحجم الدفعة الذي يكسره (نحو ٣٠ دقيقة)

مفتوح. والتوقّع في الإكمال أقل.

يُطبّع `nn.BatchNorm1d` مدخلات كل طبقة عبر الدفعة، فترى الطبقة التالية توزيعًا مستقرًا بدل توزيع
ينزاح في كل خطوة. ويشتري تدريبًا أسرع وتحمّلًا أكبر لمعدّل تعلّم سيّئ الاختيار، وأثرًا تنظيميًا خفيفًا
مجّانًا.

أضِفه وأعِد التشغيل وسجّل التغيّر. ثم اضبط `batch_size=2` وأعِد التشغيل مرة أخرى.

فعند دفعة من ٢ تكون تُطبّع بمتوسط وتباين **صورتين**، وهذا ليس توزيعًا بل ضوضاء تترنّح في كل خطوة. ودون
الثمانية تقريبًا يتوقّف تطبيع الدفعات عن النفع ويبدأ بالضرر. قِسه ولا تصدّقه، ثم اكتب الجملة الواحدة
التي تفسّر السبب، وأجب عن السؤال الختامي في الخليّة بعدها.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Three runs: baseline, batch_norm=True, and batch_norm=True with
#    batch_size=2. Keep the epoch count low — 2-image batches mean 500
#    optimiser steps per epoch, and it is the slow one.
# 2) Compare best validation loss across the three, and how many epochs each took
#    to get under 0.4.
# 3) Remember that batch norm behaves differently in train() and eval() — your
#    train function already handles that, which is why it must.
# Search: "pytorch BatchNorm1d small batch size problem"
# https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html
#
# ١) ثلاثة تشغيلات: خط الأساس، و`batch_norm=True`، و`batch_norm=True` مع
#    `batch_size=2`. وأبقِ عدد الحقب صغيرًا — فدفعات من صورتين تعني خمسمئة
#    خطوة تحسين في الحقبة، وهو البطيء.
# ٢) قارن أفضل خسارة تحقّق بين الثلاثة، وكم حقبة احتاج كل واحد للنزول
#    تحت ٠٫٤.
# ٣) وتذكّر أن تطبيع الدفعات يتصرّف تصرّفًا مختلفًا في `train()` و`eval()` — ودالة
#    تدريبك تتولّى ذلك أصلًا، ولهذا يجب أن تتولّاه.
# ابحث عن: "pytorch BatchNorm1d small batch size problem"
# ────────────────────────────────────────────────────────────────────

# TODO: Three runs: no batch norm, batch norm, batch norm at batch_size=2.
# مهمة: ثلاثة تشغيلات: بلا تطبيع دفعات، ومع تطبيع، ومع تطبيع بحجم دفعة ٢.
print(batch_norm_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

**Your closing answers**, and the second one is the week's exit question:

1. Why does batch norm degrade at `batch_size=2`, and what would you use instead if your batches had
   to be that small?
2. **Given this week's numbers** — Monday's tree on a table, Thursday's network on two coordinates,
   today's comparison on pixels — what kind of data would you reach for a neural network for, and what
   kind would you reach for a tree for?

<div dir="rtl" align="right">

**إجاباتك الختامية**، والثانية هي سؤال خروج الأسبوع:

١. لماذا يتدهور تطبيع الدفعات عند `batch_size=2`، وماذا كنت ستستخدم بدلًا منه لو اضطُرّت دفعاتك أن
   تكون بهذا الصغر؟
٢. **بالنظر إلى أرقام هذا الأسبوع** — شجرة الاثنين على جدول، وشبكة الخميس على إحداثيين، ومقارنة اليوم
   على البكسلات — لأي نوع من البيانات تمدّ يدك إلى شبكة عصبية، ولأي نوع تمدّها إلى شجرة؟

</div>


## Save your artefacts

Two files, and the first one is the capstone's requirement in miniature.

- **`experiments.parquet`** — every run, every hyperparameter, both losses, the seed. **The capstone
  brief released today asks for exactly this table.** A report that says "we tuned the model and got
  0.90" is not defensible; one that ships this file is.
- **`best_model.pt`** — the state dict of your best configuration **and** the config that produced
  it, in one file. Weights without their config are not a model; you cannot rebuild the architecture
  to load them into.

Then reload it and confirm the reloaded copy reproduces the recorded score exactly. A checkpoint
nobody has reloaded is a checkpoint that does not work yet.

<div dir="rtl" align="right">

## احفظ مخرجاتك

ملفان، والأول هو متطلّب مشروع التخرّج مصغّرًا.

- **`experiments.parquet`** وفيه كل تشغيل وكل معامل والخسارتان والبذرة. **وكرّاسة مشروع التخرّج
  المنشورة اليوم تطلب هذا الجدول بالضبط.** فالتقرير الذي يقول «ضبطنا النموذج فحصلنا على ٠٫٩٠» لا
  يمكن الدفاع عنه، والذي يُرفق هذا الملف يمكن.
- **`best_model.pt`** وفيه `state_dict` أفضل إعداد **والإعداد** الذي أنتجه، في ملف واحد. فالأوزان بلا
  إعدادها ليست نموذجًا؛ إذ لا تستطيع إعادة بناء المعمارية لتحميلها فيها.

ثم أعِد تحميله وتأكّد أن النسخة المُعاد تحميلها تُعيد إنتاج النتيجة المسجّلة بالضبط. فنقطة الحفظ التي
لم يُعِد أحد تحميلها هي نقطة حفظ لا تعمل بعد.

</div>


In [ ]:
experiments_path = ARTEFACT_DIR / "experiments.parquet"
experiments.to_parquet(experiments_path, index=False)

# The best configuration, retrained with early stopping on the original 1,000 images.
final_history, final_model, final_best_state = train(**best_config, patience=20)
final_model.load_state_dict(final_best_state)
recorded_loss, recorded_accuracy = score(final_model, X_val, y_val)

model_path = ARTEFACT_DIR / "best_model.pt"
torch.save({"state_dict": final_best_state,
            "config": best_config,
            "val_loss": recorded_loss,
            "val_accuracy": recorded_accuracy,
            "transform": {"mean": MNIST_MEAN, "std": MNIST_STD},
            "reckoning": {"network_test_accuracy": {"mean": network_mean, "std": network_std},
                          "tree_test_accuracy": {"mean": tree_mean, "std": tree_std},
                          "draws": N_DRAWS, "train_images": N_TRAIN}},
           model_path)

# Reload from scratch — a checkpoint nobody reloaded is a checkpoint that does not work.
checkpoint = torch.load(model_path)
rebuilt = build_model(checkpoint["config"]["width"], checkpoint["config"]["dropout"],
                      checkpoint["config"]["batch_norm"], checkpoint["config"]["seed"])
rebuilt.load_state_dict(checkpoint["state_dict"])
reloaded_loss, reloaded_accuracy = score(rebuilt, X_val, y_val)

reloaded_experiments = pd.read_parquet(experiments_path)
print(f"Saved {experiments_path}")
print(f"  {len(reloaded_experiments)} runs, {len(reloaded_experiments.columns)} columns")
print(f"\nSaved {model_path}")
print(f"  config: {checkpoint['config']}")
print(f"  recorded validation loss: {recorded_loss:.6f}")
print(f"  reloaded validation loss: {reloaded_loss:.6f}")
print(f"  reproduces exactly: {math.isclose(recorded_loss, reloaded_loss, abs_tol=1e-9)}")

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

The fourth one is the habit this lab exists to enforce: it walks every pair of rows in your
experiment log and fails if any two of them differ by more than one hyperparameter. The instruction
has never been enough on its own.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخليّة أخيرًا. كل فحص يفشل يخبرك بما تُصلحه ولماذا.

والفحص الرابع هو العادة التي وُجد هذا المعمل لفرضها: فهو يمرّ على كل صف من سجلّ تجاربك مقابل خط الأساس ويفشل إن
حرّك أحدها مقبضين. فالتعليمة وحدها لم تكفِ يومًا.

</div>


In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check(overfit_history.iloc[-1]["train_loss"] < 0.05
      and overfit_history.iloc[-1]["val_loss"] > overfit_history["val_loss"].min() * 1.2,
      f"task 2.1 must actually overfit: training loss below 0.05 (got "
      f"{overfit_history.iloc[-1]['train_loss']:.4f}) and a final validation loss well "
      f"above the best one ({overfit_history.iloc[-1]['val_loss']:.4f} vs "
      f"{overfit_history['val_loss'].min():.4f})",
      f"يجب أن تُفرِط المهمة ٢٫١ في المطابقة فعلًا: خسارة تدريب دون ٠٫٠٥ (والناتج "
      f"{overfit_history.iloc[-1]['train_loss']:.4f}) وخسارة تحقّق نهائية أعلى بوضوح من "
      f"أفضلها ({overfit_history.iloc[-1]['val_loss']:.4f} مقابل "
      f"{overfit_history['val_loss'].min():.4f})")

check(train_worse,
      f"dropout must make the TRAINING loss worse — that is the regularisation doing its "
      f"job. Without dropout {comparison.iloc[0]['final_train']:.6f}, with dropout "
      f"{comparison.iloc[1]['final_train']:.6f}. (Whether it improved validation is the "
      f"experiment.)",
      f"يجب أن يُسيء Dropout خسارة **التدريب** — فهذا هو التنظيم يعمل عمله. فبلا Dropout "
      f"{comparison.iloc[0]['final_train']:.6f} ومعه {comparison.iloc[1]['final_train']:.6f}. "
      f"(أما هل حسّن التحقّق فتلك هي التجربة.)")

check(same_start,
      f"all four optimiser runs must start from identical weights, or you are comparing "
      f"initialisations. Their training losses before the first step were "
      f"{optimiser_table['loss_before_training'].round(6).tolist()}",
      f"يجب أن تبدأ تشغيلات المُحسِّنات الأربعة من أوزان متطابقة، وإلا فأنت تقارن تهيئات. "
      f"وخسائرها قبل الخطوة الأولى كانت "
      f"{optimiser_table['loss_before_training'].round(6).tolist()}")

hyperparameters = list(BASE)
baseline_row = experiments[experiments["changed"] == "-"].iloc[0]
offenders = []
for _, row in experiments.iterrows():
    differing = [k for k in hyperparameters if row[k] != baseline_row[k]]
    if len(differing) > 1:
        offenders.append((row["run"], differing))
check(len(experiments) >= 8 and not offenders,
      f"experiments.parquet needs at least 8 rows (has {len(experiments)}) and every row "
      f"must differ from the baseline by at most one hyperparameter. Offending runs: "
      f"{offenders[:3]}",
      f"يحتاج `experiments.parquet` ثمانية صفوف على الأقل (الموجود {len(experiments)}) "
      f"وأن يختلف كل صف عن خط الأساس بمعامل واحد على الأكثر. والتشغيلات المخالفة: "
      f"{offenders[:3]}")

check(restored_loss <= final_loss,
      f"early stopping must restore weights that score at least as well as the final "
      f"epoch's — restored {restored_loss:.4f} vs final {final_loss:.4f}. If the restored "
      f"weights are worse, you did not call load_state_dict(best_state).",
      f"يجب أن يستعيد الإيقاف المبكر أوزانًا لا تقلّ نتيجتها عن الحقبة الأخيرة — المستعادة "
      f"{restored_loss:.4f} مقابل النهائية {final_loss:.4f}. فإن كانت المستعادة أسوأ فأنت "
      f"لم تنادِ `load_state_dict(best_state)`.")

check(math.isclose(recorded_loss, reloaded_loss, abs_tol=1e-9),
      f"best_model.pt must reload and reproduce its recorded score exactly — recorded "
      f"{recorded_loss:.8f}, reloaded {reloaded_loss:.8f}",
      f"يجب أن يُعاد تحميل `best_model.pt` وأن يُعيد إنتاج نتيجته المسجّلة بالضبط — المسجّلة "
      f"{recorded_loss:.8f} والمُعاد تحميلها {reloaded_loss:.8f}")

check(len(network_scores) == N_DRAWS and 0.80 < network_mean < 0.99 and network_std > 0
      and "reckoning" in checkpoint,
      f"the network-versus-tree comparison must cover {N_DRAWS} draws and be recorded with "
      f"both spreads: network {network_mean:.4f} +/- {network_std:.4f}, tree "
      f"{tree_mean:.4f} +/- {tree_std:.4f}. Above 0.99 on 1,000 images means the test set "
      f"leaked into training.",
      f"يجب أن تغطّي مقارنة الشبكة بالشجرة {N_DRAWS} سحبات وأن تُسجَّل بمدى تفاوت كلٍّ منهما: "
      f"الشبكة {network_mean:.4f} ± {network_std:.4f}، والشجرة {tree_mean:.4f} ± {tree_std:.4f}. "
      f"وما فوق ٠٫٩٩ بألف صورة يعني أن مجموعة الاختبار تسرّبت إلى التدريب.")

report()

## What's next

That is week 3. You arrived able to call `.fit()` and you leave having built the thing `.fit()` calls
out of bare tensors, proved its gradients against a numerical check, rebuilt it from PyTorch's layers,
trained it on images, and then measured it honestly against a model with no neurons in it.

The last part is the one worth carrying. **On Monday's table the boosted tree won; on today's pixels,
look at your own two numbers** — and your report says which way it went and by how much. Put the two
results side by side and you have the week's actual finding: what kind of data a network is for.

Next week (**W4D1**) the data stays images and the model finally stops throwing their structure away.
`nn.Flatten` discarded which pixels were neighbours; a convolutional network is built on exactly that.
Keep today's test accuracy: it is the number the week-4 CNN has to beat on the same digits.

Three things travel with you:

- **The five-line loop**, unchanged, for every model in weeks 4 to 7.
- **`experiments.parquet`** — the shape of the log your capstone report needs. The brief is out today;
  read it tonight and start the log on day one, not the night before submission.
- **The diagnosis vocabulary.** Divergence, a flat curve at `ln k`, a widening gap. Those three
  readings apply unchanged to every model you will train from here.

<div dir="rtl" align="right">

## ماذا بعد

هذا هو الأسبوع الثالث. وصلت وأنت تعرف مناداة `.fit()` وتخرج وقد بنيت ما تناديه `.fit()` من تنسورات
عارية، وأثبتّ اشتقاقاته مقابل فحص عددي، وأعدت بناءه من طبقات PyTorch، ودرّبته على صور، ثم قِسته قياسًا
صادقًا مقابل نموذج لا عصبون فيه.

والجزء الأخير هو الجدير بالحمل. **فعلى جدول الاثنين فازت الشجرة المعزَّزة؛ وعلى بكسلات اليوم، انظر في
رقميك أنت** — وتقريرك يقول في أي اتجاه ذهب وبكم. ضَع النتيجتين جنبًا إلى جنب فتحصل على نتيجة الأسبوع
الفعلية: لأي نوع من البيانات تصلح الشبكة.

وفي الأسبوع القادم (**الأسبوع ٤ اليوم ١**) تبقى البيانات صورًا ويكفّ النموذج أخيرًا عن رمي بنيتها. فقد
رمى `nn.Flatten` معرفة أي البكسلات متجاورة، والشبكة الالتفافية مبنيّة على ذلك بالضبط. واحتفظ بدقة اختبار
اليوم: فهي الرقم الذي يجب أن تتجاوزه شبكة الأسبوع الرابع الالتفافية على الأرقام نفسها.

وثلاثة أمور تسافر معك:

- **حلقة الأسطر الخمسة** بلا تغيير، لكل نموذج في الأسابيع من الرابع إلى السابع.
- **`experiments.parquet`** وهو شكل السجلّ الذي يحتاجه تقرير مشروع تخرّجك. والكرّاسة صدرت اليوم،
  فاقرأها الليلة وابدأ السجلّ من اليوم الأول لا في ليلة التسليم.
- **مفردات التشخيص:** التباعد، والمنحنى المسطّح عند `ln k`، والفجوة المتّسعة. وهذه القراءات الثلاث
  تنطبق بلا تغيير على كل نموذج تدرّبه من هنا.

</div>
